In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Say we have an input x of batch size 4, seq_len 8 tokens and each token has emb_dim as 3
The goal is to encode the information from the past tokens for a perticular token

One of the simplest ways to do it is by taking the avg of all prev tokens at every point in context for each batch
we avg across the emb_dim

But just doing a simple avg is not an efficient way of encoding the information from the past tokens.


In [ ]:
# Approach 1, simple for loop
torch.manual_seed(1356)
B,CL,ED = 4,8,3
x = torch.randn(B,CL,ED)
x_new = torch.zeros_like(x)
print(x.shape)
# print(x_new.shape)

for b in range(B):
    for c in range(CL):
        xp = x[b, :c+1] # (c, ED) for the batch take all the tokens till the current token
        x_new[b, c] = torch.mean(xp, dim=0) # take the mean of all the tokens till the current token vertically (across the sequence dimension, i.e avg of each emb dim across seq_len)

print("x[0]\n", x[0])
print("x_new[0]\n", x_new[0])

# Now in each batch, for every token 't' in the context its past token info is encoded.

torch.Size([4, 8, 3])
x[0]
 tensor([[-2.2394, -0.3135,  0.3673],
        [-0.3950, -1.6581, -2.2587],
        [-0.0946, -0.2392, -0.4104],
        [-1.5983, -1.0144, -0.3037],
        [ 0.8711, -0.2735,  0.1410],
        [-2.5288,  2.0586, -1.5377],
        [ 0.1990,  1.0479, -0.1373],
        [-0.3055, -0.4565,  0.1797]])
x_new[0]
 tensor([[-2.2394, -0.3135,  0.3673],
        [-1.3172, -0.9858, -0.9457],
        [-0.9097, -0.7370, -0.7673],
        [-1.0818, -0.8063, -0.6514],
        [-0.6912, -0.6998, -0.4929],
        [-0.9975, -0.2400, -0.6670],
        [-0.8266, -0.0560, -0.5914],
        [-0.7614, -0.1061, -0.4950]])


* One math tric we can use is if we multiply a tensor of all onces with the 'x' tensor we get sum of all elements across columns (seq_len dim, i.e sum of emb_dim for each seq_len).
* If instead of all ones we have a lower triangular lower matrix we get the sum incrementally
* now instead of ones in the lower triangular if we have values such that each row sums up to 1, we get the weighted average of the past tokens for each token.

In [ ]:
tl = torch.tril(torch.ones(CL, CL))
w = tl / tl.sum(dim=1, keepdim=True)
print(w)

# This is same as taking avg of all previous tokens for each token across emb_dim
x_new1 = w @ x

print(torch.allclose(x_new, x_new1))
print(x_new[0])


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
True
tensor([[-2.2394, -0.3135,  0.3673],
        [-1.3172, -0.9858, -0.9457],
        [-0.9097, -0.7370, -0.7673],
        [-1.0818, -0.8063, -0.6514],
        [-0.6912, -0.6998, -0.4929],
        [-0.9975, -0.2400, -0.6670],
        [-0.8266, -0.0560, -0.5914],
        [-0.7614, -0.1061, -0.4950]])


* One other efficient way we can do this is by softmax

In [ ]:
tl = torch.tril(torch.ones(CL, CL))
w = torch.zeros(CL, CL)
w = w.masked_fill(tl == 0, float('-inf')) # maskes out the upper tri part with -inf
w = F.softmax(w, dim=-1) # This is equvivalent to "w = tl / tl.sum(dim=1, keepdim=True)"

x_new2 = w @ x
print(torch.allclose(x_new, x_new2))


True


## CRUX of self attn
* In the above approach, we have taken simple avg of all past tokens for each batch in x, this is done by making the W as a tril and make all its elements sum to 1 in each row.
* But this uniformity will not make the model perform well. We need 'w' to be non -uniform but dependent on the data (past input)
* To achieve this, each token input would emit two vectors,  Query Q and Key K
* Query specifies what the current token is looking for (i,e is it looking for a noun or ad-verb etc..)
* Key specifies what the current token has (i.e is it a noun or adverb etc.. this is a very very rough example though)
* Qw,Kw vectors weights are initialized as (ED, ED), and the input x is passed through these vectors to get Q and K of size (CL, ED)
* Now we do Q@K.T, this will do a dot product b/w all keys and all queries. Resulting in (CL, CL) matrix
* Each token row has scores for every other token in the context, tokens which are more closely associated with each other will have higher dot product. This will becore our `w`
* What we achieved is instead of making it all zeros (uniform affinity) we are learning the affinity via Q,K (by gradient descent)
* Now we still mask the `W` or (attn weights) and make it a tril
* Then we apply softmax on `W` to attn scores, high dot products becomes high attn scores


In [ ]:
B,CL,ED = 4,8,32
x = torch.randn(B,CL,ED)

head_size = 16
Qw = nn.Linear(ED, head_size)
Kw = nn.Linear(ED, head_size)

Q = Qw(x) #(B, CL, 16)
K = Kw(x) #(B, CL, 16)

# print(K.transpose(1, 2).shape) [4, 16, 8])

w = Q @ K.transpose(1, 2) # (B, CL, 16) @ (B, 16, CL) = (B, CL, CL)

tl = torch.tril(torch.ones(CL, CL))
w = w.masked_fill(tl == 0, float('-inf')) # maskes out the upper tri part with -inf
w = F.softmax(w, dim=-1) # This is equvivalent to "w = tl / tl.sum(dim=1, keepdim=True)"

out = w @ x

torch.Size([4, 16, 8])


* But instead of adirectly aggregating (multiplying) the attn scores with the inout x `out = w @ x`, we create another matrix from x called value matrix V
* We can think of `Q,K,V` as teh public versions of the `x` which represent different information
* Q representing what the current token is looking for
* K representing what the current token has
* V representing what the current token has that is useful
* These vecors are what used in the atten block and the final outout is again added back to x as a residual connection


In [35]:
B,CL,ED = 4,8,32
x = torch.randn(B,CL,ED)

head_size = 16
Qw = nn.Linear(ED, head_size)
Kw = nn.Linear(ED, head_size)
Vw = nn.Linear(ED, head_size)

Q = Qw(x) #(B, CL, 16)
K = Kw(x) #(B, CL, 16)
V = Vw(x) #(B, CL, 16)

# print(K.transpose(1, 2).shape) [4, 16, 8])

w = Q @ K.transpose(1, 2) # (B, CL, 16) @ (B, 16, CL) = (B, CL, CL)
w = w * (head_size ** -0.5)

tl = torch.tril(torch.ones(CL, CL))
w = w.masked_fill(tl == 0, float('-inf')) # maskes out the upper tri part with -inf
w = F.softmax(w, dim=-1) # This is equvivalent to "w = tl / tl.sum(dim=1, keepdim=True)"

out = w @ V # (B, CL, CL) @ (B, CL, 16) = (B, CL, 16)

* Attention is a communication mechanism
* There is no notion of position in attention, hence we add pos embiddings
* One importent thing is before applying softmax to our attention weigths `w` we need to scale them by 1/sqert(attn_head_size)
    * The reason being, as the head size increases the dot products also increase and when we apply softmax to them they become very spiky and have very small gradients. So we scale them down to keep them in the linear region of softmax.
    * Also if the dt product of a perticular key and query have large value, the softmax will push other values to 0, making the attention too focused on that one key-query pair.